# LABORATORIO 6


📋 Instrucciones Generales
● Desarrolle las soluciones en celdas de código independientes dentro de su entorno de
Google Colab.
● Documente su código explicando la lógica detrás de sus funciones.
● Prohibido el uso de bucles for para evaluar filas individuales. Todo el procesamiento
matemático condicional debe hacerse con np.where.
● Utilice la función display() para mostrar las tablas resultantes.

🏢 Caso de Negocio: Algoritmo de Bonificación y
Reportes Automatizados
Usted es el Data Engineer Senior de TechMarket, un retail multinacional. El equipo de Recursos
Humanos le ha enviado la base de datos del desempeño trimestral de su fuerza de ventas.
Su misión tiene tres etapas:
1. Crear un algoritmo vectorizado que clasifique el rendimiento y calcule bonos
automáticamente.
2. Generar un resumen gerencial (agrupado) para el Directorio.
3. Automatizar la creación de reportes individuales por región.
📥 Preparación del Entorno (Ejecute esta celda primero)
Copie y ejecute este código en su cuaderno para generar el DataFrame sobre el que trabajará.

In [34]:
import pandas as pd
import numpy as np
# Generación de la data de desempeño
data_desempeno = {
'id_empleado': ['E01', 'E02', 'E03', 'E04', 'E05', 'E06',
'E07', 'E08'],
'region': ['Norte', 'Norte', 'Sur', 'Sur', 'Centro', 'Norte',
'Sur', 'Centro'],
'ventas_usd': [12000, 4500, 8000, 15000, 3000, np.nan, 9500,
11000],
'satisfaccion_cliente': [4.8, 3.2, 4.1, 4.9, 2.5, 4.0, 3.8,
4.7],
'meses_antiguedad': [24, 6, 12, 36, 4, 18, 9, 48]
}
df_ventas = pd.DataFrame(data_desempeno)
display(df_ventas.head())

,id_empleado,region,ventas_usd,satisfaccion_cliente,meses_antiguedad
0,E01,Norte,12000.0,4.8,24
1,E02,Norte,4500.0,3.2,6
2,E03,Sur,8000.0,4.1,12
3,E04,Sur,15000.0,4.9,36
4,E05,Centro,3000.0,2.5,4


🚀 Parte 1: El Motor de Reglas (np.where Avanzado)
El directorio ha establecido reglas estrictas para evaluar a los empleados y asignarles su bono
trimestral. Debe crear dos nuevas columnas en df_ventas aplicando lógica condicional.
Misiones:
1. Columna categoria_rendimiento: Utilice np.where anidado para clasificar a cada
empleado:
○ Si sus ventas_usd son mayores a 10,000 Y su satisfaccion_cliente es
mayor o igual a 4.5 -> "Top Talent".
○ Si sus ventas_usd son mayores o iguales a 5,000 -> "Regular".
○ Si no cumple ninguna de las anteriores (o si las ventas son nulas) -> "Bajo
Rendimiento".

2. Columna bono_usd: Utilice np.where para calcular matemáticamente el bono:
○ Si es "Top Talent", el bono es el 15% de sus ventas_usd.
○ Si es "Regular", el bono es el 5% de sus ventas_usd.
○ Si es "Bajo Rendimiento", el bono es 0.

In [35]:
# Como falta 1 dato en las ventas asumi un promedio
df_ventas["ventas_usd"] = df_ventas["ventas_usd"].fillna(df_ventas["ventas_usd"].mean())

df_ventas["categoria_rendimiento"] = np.where( (df_ventas["ventas_usd"] > 10000) & (df_ventas["satisfaccion_cliente"] >= 4.5), "Top Talent", np.where(df_ventas["ventas_usd"] >= 5000 , "Regular", "Bajo Rendimiento") )

In [36]:
display( df_ventas)

,id_empleado,region,ventas_usd,satisfaccion_cliente,meses_antiguedad,categoria_rendimiento
0,E01,Norte,12000.0,4.8,24,Top Talent
1,E02,Norte,4500.0,3.2,6,Bajo Rendimiento
2,E03,Sur,8000.0,4.1,12,Regular
3,E04,Sur,15000.0,4.9,36,Top Talent
4,E05,Centro,3000.0,2.5,4,Bajo Rendimiento
5,E06,Norte,9000.0,4.0,18,Regular
6,E07,Sur,9500.0,3.8,9,Regular
7,E08,Centro,11000.0,4.7,48,Top Talent


In [37]:
df_ventas["bono_usd"] = np.where(df_ventas["categoria_rendimiento"] == "Top Talent", df_ventas["ventas_usd"] * 0.15, np.where(df_ventas["categoria_rendimiento"] == "Regular",df_ventas["ventas_usd"] * 0.05 , df_ventas["ventas_usd"] * 0 ))

In [38]:
display( df_ventas)

,id_empleado,region,ventas_usd,satisfaccion_cliente,meses_antiguedad,categoria_rendimiento,bono_usd
0,E01,Norte,12000.0,4.8,24,Top Talent,1800.0
1,E02,Norte,4500.0,3.2,6,Bajo Rendimiento,0.0
2,E03,Sur,8000.0,4.1,12,Regular,400.0
3,E04,Sur,15000.0,4.9,36,Top Talent,2250.0
4,E05,Centro,3000.0,2.5,4,Bajo Rendimiento,0.0
5,E06,Norte,9000.0,4.0,18,Regular,450.0
6,E07,Sur,9500.0,3.8,9,Regular,475.0
7,E08,Centro,11000.0,4.7,48,Top Talent,1650.0


In [39]:
df_resumen_gerencial = df_ventas.groupby(['region', 'categoria_rendimiento'], as_index=False).agg(
    ventas_usd=('ventas_usd', 'sum'),
    bono_usd=('bono_usd', 'sum'),
    satisfaccion_cliente=('satisfaccion_cliente', 'mean'),
    id_empleado=('id_empleado', 'count')
)
display(df_resumen_gerencial)

,region,categoria_rendimiento,ventas_usd,bono_usd,satisfaccion_cliente,id_empleado
0,Centro,Bajo Rendimiento,3000.0,0.0,2.50,1
1,Centro,Top Talent,11000.0,1650.0,4.70,1
2,Norte,Bajo Rendimiento,4500.0,0.0,3.20,1
3,Norte,Regular,9000.0,450.0,4.00,1
4,Norte,Top Talent,12000.0,1800.0,4.80,1
5,Sur,Regular,17500.0,875.0,3.95,2
6,Sur,Top Talent,15000.0,2250.0,4.90,1


In [40]:
# Misión 1: Generar el objeto agrupado únicamente por la región
grupos_por_region = df_ventas.groupby('region')

# Misión 2: Bucle for combinando enumerate (iniciando en 1) para iterar sobre los grupos
for i, (nombre_region, datos_region) in enumerate(grupos_por_region, start=1):

    # Misión 3: Imprimir el texto dinámico en consola
    print(f"Generando Reporte {i} - Región: {nombre_region}")

    # Misión 4: Imprimir el mini DataFrame mostrando únicamente las columnas solicitadas
    display(datos_region[['id_empleado', 'categoria_rendimiento', 'bono_usd']])

    # Separador visual
    print("\n" + "="*40 + "\n")

Generando Reporte 1 - Región: Centro


,id_empleado,categoria_rendimiento,bono_usd
4,E05,Bajo Rendimiento,0.0
7,E08,Top Talent,1650.0




Generando Reporte 2 - Región: Norte


,id_empleado,categoria_rendimiento,bono_usd
0,E01,Top Talent,1800.0
1,E02,Bajo Rendimiento,0.0
5,E06,Regular,450.0




Generando Reporte 3 - Región: Sur


,id_empleado,categoria_rendimiento,bono_usd
2,E03,Regular,400.0
3,E04,Top Talent,2250.0
6,E07,Regular,475.0
